# 91 QC nodal shot catalog and metadata matches

This notebook performs the first QC pass after `90_nodal_fullnode_detection_pick_and_shotgathers_UPDATED.ipynb` has finished.

It covers:

1. **Catalog QC**: event counts, receiver counts, file existence, pick counts, missing stations, and database integrity checks.
2. **Shot-gather inspection**: load saved SEG-Y/MiniSEED/PNG products and make representative wiggle plots.
3. **Metadata match QC**: compare nodal detections with Geode/streamer metadata imported by `89_*`.

The notebook does not modify the catalog. It is read-only except for optional CSV/PNG exports under `qc_outputs/`.

In [ ]:
from pathlib import Path
import sqlite3
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, UTCDateTime

# Optional: add local repo lib directory for segy_tools
import sys

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
REPO_ROOT = Path('/Volumes/tachyon/karstgeo')  # edit if needed
LIB_DIR = REPO_ROOT / 'lib'
if LIB_DIR.exists():
    sys.path.append(str(LIB_DIR.resolve()))

try:
    from segy_tools.gather import plot_wiggle_gather_from_stream, plot_wiggle_gather_from_segy
    HAVE_SEGY_TOOLS = True
except Exception as e:
    HAVE_SEGY_TOOLS = False
    print('Could not import segy_tools. Some plotting cells will use fallback plotting only.')
    print(e)

CATALOG_DB = PROJECT_ROOT / 'catalog' / 'lbssp_shot_catalog.sqlite'
OUTPUT_ROOT = PROJECT_ROOT / 'nodal_fullnode_shotgathers_v3'
QC_OUTPUT = OUTPUT_ROOT / 'qc_outputs'
QC_OUTPUT.mkdir(parents=True, exist_ok=True)

print('CATALOG_DB:', CATALOG_DB)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('QC_OUTPUT:', QC_OUTPUT)
print('HAVE_SEGY_TOOLS:', HAVE_SEGY_TOOLS)

## 1. Open catalog and load tables

This section reads the SQLite database into pandas DataFrames. It does not write to the database.

In [ ]:
assert CATALOG_DB.exists(), f'Catalog database not found: {CATALOG_DB}'
conn = sqlite3.connect(CATALOG_DB)
conn.row_factory = sqlite3.Row

def list_tables(conn):
    return pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn,
    )['name'].tolist()

tables = list_tables(conn)
print('Tables:', tables)

for table in tables:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn)['n'].iloc[0]
    print(f'{table:25s} {n:10d}')

In [ ]:
def read_table(name):
    if name not in tables:
        print(f'Table missing: {name}')
        return pd.DataFrame()
    return pd.read_sql(f'SELECT * FROM {name}', conn)

processing_runs = read_table('processing_runs')
shot_events = read_table('shot_events')
receiver_geometry = read_table('receiver_geometry')
shot_gather_files = read_table('shot_gather_files')
trace_index = read_table('trace_index')
picks = read_table('picks')

# Parse datetimes where present
for df in [shot_events, trace_index, picks, processing_runs]:
    for col in df.columns:
        if col.endswith('_utc') or col in ['shot_time_utc', 'detection_time_utc', 'pick_time_utc', 'run_time_utc']:
            df[col + '_dt'] = pd.to_datetime(df[col], errors='coerce', utc=True)

print('Loaded tables.')

## 2. Catalog QC summary

Check the event counts, metadata matches, file products, trace counts, and pick counts.

In [ ]:
def display_if_not_empty(df, title=None, n=20):
    if title:
        print('
' + title)
    if df is None or df.empty:
        print('(empty)')
    else:
        display(df.head(n))

# Event counts by instrument/system/time window
cols = [c for c in ['instrument_system', 'line', 'timewindow_label', 'source_type', 'metadata_match_status'] if c in shot_events.columns]
print('shot_events columns used for grouping:', cols)
if cols:
    display(shot_events.groupby(cols, dropna=False).size().reset_index(name='n').sort_values('n', ascending=False).head(50))

# Nodal events only
nodal_events = shot_events[shot_events.get('instrument_system', '').eq('nodal')] if 'instrument_system' in shot_events.columns else shot_events.iloc[0:0]
print('Nodal events:', len(nodal_events))
if 'timewindow_label' in nodal_events.columns:
    display(nodal_events.groupby('timewindow_label', dropna=False).size().reset_index(name='n_events'))

In [ ]:
# File products by time window / component / file type
if not shot_gather_files.empty:
    group_cols = [c for c in ['instrument_system', 'timewindow_label', 'component', 'file_type', 'processing_level'] if c in shot_gather_files.columns]
    display(shot_gather_files.groupby(group_cols, dropna=False).size().reset_index(name='n_files').sort_values(group_cols))

    # Does every registered file exist?
    shot_gather_files['exists'] = shot_gather_files['file_path'].apply(lambda p: Path(str(p)).exists())
    display(shot_gather_files['exists'].value_counts(dropna=False).rename_axis('exists').reset_index(name='n'))
    missing_files = shot_gather_files[~shot_gather_files['exists']]
    if len(missing_files):
        display(missing_files.head(20))
        missing_files.to_csv(QC_OUTPUT / 'missing_registered_files.csv', index=False)
else:
    print('shot_gather_files table is empty.')

In [ ]:
# Trace count QC by event
if not trace_index.empty:
    trace_summary = (
        trace_index
        .groupby('event_id', dropna=False)
        .agg(
            n_trace_rows=('event_id', 'size'),
            n_stations=('station', 'nunique') if 'station' in trace_index.columns else ('event_id', 'size'),
            n_channels=('channel', 'nunique') if 'channel' in trace_index.columns else ('event_id', 'size'),
            min_receiver_x_m=('receiver_x_m', 'min') if 'receiver_x_m' in trace_index.columns else ('event_id', 'size'),
            max_receiver_x_m=('receiver_x_m', 'max') if 'receiver_x_m' in trace_index.columns else ('event_id', 'size'),
        )
        .reset_index()
    )
    display(trace_summary.describe(include='all'))
    display(trace_summary.sort_values('n_stations').head(20))
    trace_summary.to_csv(QC_OUTPUT / 'trace_count_summary_by_event.csv', index=False)
else:
    print('trace_index table is empty.')

In [ ]:
# Pick count QC by event and component
if not picks.empty:
    pick_group_cols = [c for c in ['event_id', 'component', 'phase', 'picker'] if c in picks.columns]
    pick_summary = picks.groupby('event_id', dropna=False).size().reset_index(name='n_picks')
    display(pick_summary.describe(include='all'))
    display(pick_summary.sort_values('n_picks').head(20))

    if 'pick_quality' in picks.columns:
        display(picks['pick_quality'].value_counts(dropna=False).rename_axis('pick_quality').reset_index(name='n'))

    if 'component' in picks.columns:
        display(picks.groupby(['component'], dropna=False).size().reset_index(name='n_picks'))

    pick_summary.to_csv(QC_OUTPUT / 'pick_count_summary_by_event.csv', index=False)
else:
    print('picks table is empty.')

## 3. Receiver geometry QC

Check that receiver positions look sensible and that the number of available nodal stations is as expected for each geometry/time window.

In [ ]:
if not receiver_geometry.empty:
    display(receiver_geometry.head())
    geom_cols = [c for c in ['instrument_system', 'line', 'timewindow_label', 'geometry_id'] if c in receiver_geometry.columns]
    if geom_cols:
        display(receiver_geometry.groupby(geom_cols, dropna=False).agg(
            n_rows=('station', 'size') if 'station' in receiver_geometry.columns else ('receiver_x_m', 'size'),
            n_stations=('station', 'nunique') if 'station' in receiver_geometry.columns else ('receiver_x_m', 'size'),
            min_x=('receiver_x_m', 'min') if 'receiver_x_m' in receiver_geometry.columns else ('station', 'size'),
            max_x=('receiver_x_m', 'max') if 'receiver_x_m' in receiver_geometry.columns else ('station', 'size'),
        ).reset_index())

    if 'receiver_x_m' in receiver_geometry.columns:
        fig, ax = plt.subplots(figsize=(12, 3))
        rg = receiver_geometry.dropna(subset=['receiver_x_m']).copy()
        rg = rg.sort_values('receiver_x_m')
        ax.plot(rg['receiver_x_m'], np.zeros(len(rg)), '|', markersize=20)
        ax.set_xlabel('Receiver x (m)')
        ax.set_yticks([])
        ax.set_title('Receiver positions in catalog')
        ax.grid(True, axis='x', alpha=0.3)
        plt.show()
else:
    print('receiver_geometry table is empty.')

## 4. Metadata match QC

This compares nodal detections with metadata imported from the Geode/field-note workbook. The most useful fields are `matched_metadata_event_id`, `metadata_match_status`, `source_x_m`, `source_type`, `operator`, `plate_type`, and `n_blows` if present.

In [ ]:
if not shot_events.empty:
    nodal = shot_events[shot_events.get('instrument_system', '').eq('nodal')].copy() if 'instrument_system' in shot_events.columns else pd.DataFrame()
    non_nodal = shot_events[~shot_events.get('instrument_system', '').eq('nodal')].copy() if 'instrument_system' in shot_events.columns else pd.DataFrame()

    print('Nodal events:', len(nodal))
    print('Non-nodal metadata events:', len(non_nodal))

    for col in ['metadata_match_status', 'source_type', 'operator', 'plate_type', 'line', 'timewindow_label']:
        if col in nodal.columns:
            print('
', col)
            display(nodal[col].value_counts(dropna=False).rename_axis(col).reset_index(name='n'))

    # Show matched event examples
    match_cols = [c for c in [
        'event_id', 'line', 'timewindow_label', 'detection_time_utc', 'shot_time_utc',
        'source_x_m', 'source_type', 'operator', 'plate_type', 'n_blows',
        'metadata_match_status', 'matched_metadata_event_id', 'metadata_time_residual_s'
    ] if c in nodal.columns]

    if match_cols:
        display(nodal[match_cols].head(30))
        nodal[match_cols].to_csv(QC_OUTPUT / 'nodal_metadata_match_qc.csv', index=False)
else:
    print('shot_events table is empty.')

In [ ]:
# Histogram of time residuals for metadata matches, if available
if 'metadata_time_residual_s' in shot_events.columns:
    nodal = shot_events[shot_events.get('instrument_system', '').eq('nodal')].copy() if 'instrument_system' in shot_events.columns else shot_events.copy()
    vals = pd.to_numeric(nodal['metadata_time_residual_s'], errors='coerce').dropna()
    if len(vals):
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(vals, bins=50)
        ax.set_xlabel('metadata_time_residual_s')
        ax.set_ylabel('Count')
        ax.set_title('Nodal detection time minus matched metadata time')
        ax.grid(True, alpha=0.3)
        plt.show()
        print(vals.describe())
    else:
        print('No metadata_time_residual_s values found.')
else:
    print('No metadata_time_residual_s column found.')

## 5. Select representative events for visual QC

Choose a few nodal events to inspect. By default this selects the first few events per time window, and also a few metadata-matched events if available.

In [ ]:
def choose_qc_events(shot_events, n_per_window=3, include_matched=True):
    if shot_events.empty or 'instrument_system' not in shot_events.columns:
        return []
    nodal = shot_events[shot_events['instrument_system'].eq('nodal')].copy()
    if nodal.empty:
        return []

    sort_cols = [c for c in ['timewindow_label', 'detection_time_utc', 'event_id'] if c in nodal.columns]
    if sort_cols:
        nodal = nodal.sort_values(sort_cols)

    selected = []
    if 'timewindow_label' in nodal.columns:
        for tw, sub in nodal.groupby('timewindow_label', dropna=False):
            selected.extend(sub.head(n_per_window)['event_id'].tolist())
    else:
        selected.extend(nodal.head(n_per_window)['event_id'].tolist())

    if include_matched and 'metadata_match_status' in nodal.columns:
        matched = nodal[nodal['metadata_match_status'].astype(str).str.contains('match', case=False, na=False)]
        selected.extend(matched.head(10)['event_id'].tolist())

    # unique, preserve order
    seen = set()
    out = []
    for e in selected:
        if e not in seen:
            out.append(e)
            seen.add(e)
    return out

QC_EVENT_IDS = choose_qc_events(shot_events, n_per_window=3)
print('Selected QC events:', len(QC_EVENT_IDS))
print(QC_EVENT_IDS[:20])

## 6. Inspect registered PNG outputs

This is the fastest way to visually inspect many gathers without rereading waveform data.

In [ ]:
from IPython.display import Image, display as ipy_display

def show_registered_pngs(event_id, component='Z', max_images=4):
    if shot_gather_files.empty:
        print('No shot_gather_files table.')
        return
    sub = shot_gather_files[shot_gather_files['event_id'].eq(event_id)].copy()
    if 'component' in sub.columns:
        sub = sub[sub['component'].eq(component)]
    if 'file_type' in sub.columns:
        sub = sub[sub['file_type'].astype(str).str.contains('png', case=False, na=False)]
    sub = sub.head(max_images)
    if sub.empty:
        print(f'No PNGs found for {event_id}, component={component}')
        return
    for _, row in sub.iterrows():
        path = Path(row['file_path'])
        print(path)
        if path.exists():
            ipy_display(Image(filename=str(path)))
        else:
            print('Missing:', path)

for eid in QC_EVENT_IDS[:3]:
    print('
EVENT', eid)
    show_registered_pngs(eid, component='Z', max_images=2)

## 7. Load and plot selected SEG-Y or MiniSEED gathers

This section rereads waveform files and plots them. It prefers SEG-Y Z-component files when available, otherwise it falls back to the 3C MiniSEED gather.

In [ ]:
def get_gather_file(event_id, component='Z', preferred_file_type='segy'):
    sub = shot_gather_files[shot_gather_files['event_id'].eq(event_id)].copy()
    if sub.empty:
        return None
    if 'component' in sub.columns:
        sub_comp = sub[sub['component'].eq(component)]
    else:
        sub_comp = sub
    if 'file_type' in sub_comp.columns:
        sub_pref = sub_comp[sub_comp['file_type'].eq(preferred_file_type)]
        if not sub_pref.empty:
            return Path(sub_pref.iloc[0]['file_path'])
    # fallback to 3C mseed
    if 'file_type' in sub.columns:
        sub_mseed = sub[sub['file_type'].eq('mseed')]
        if not sub_mseed.empty:
            return Path(sub_mseed.iloc[0]['file_path'])
    return Path(sub.iloc[0]['file_path'])

def simple_wiggle_from_stream(st, title='', tmin=0.0, tmax=0.5, component=None, normalize=True):
    if component is not None:
        st = st.select(channel=f'*{component}')
    if len(st) == 0:
        print('No traces to plot')
        return None
    st = st.copy()
    st.sort(keys=['station', 'channel'])
    npts = min(tr.stats.npts for tr in st)
    dt = st[0].stats.delta
    t = np.arange(npts) * dt
    mask = (t >= tmin) & (t <= tmax)
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, tr in enumerate(st):
        x = tr.data[:npts].astype(float)
        x = x - np.nanmedian(x)
        if normalize:
            s = np.nanpercentile(np.abs(x), 99)
            if s > 0:
                x = x / s
        ax.plot(x[mask] * 0.4 + i, t[mask], linewidth=0.7)
    ax.invert_yaxis()
    ax.set_xlabel('Trace index + normalized amplitude')
    ax.set_ylabel('Time since file start (s)')
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.show()
    return fig

def plot_event_gather(event_id, component='Z', tmin=0.0, tmax=0.5):
    path = get_gather_file(event_id, component=component, preferred_file_type='segy')
    print('event_id:', event_id)
    print('path:', path)
    if path is None or not path.exists():
        print('No file found')
        return
    if path.suffix.lower() in ['.sgy', '.segy'] and HAVE_SEGY_TOOLS:
        return plot_wiggle_gather_from_segy(
            path,
            sort_by='receiver_x',
            title=f'{event_id} {component}',
            tmin=tmin,
            tmax=tmax,
            scale=0.8,
            clip_percentile=99,
            normalize=True,
        )
    else:
        st = read(str(path))
        return simple_wiggle_from_stream(st, title=f'{event_id} {component}', tmin=tmin, tmax=tmax, component=component)

for eid in QC_EVENT_IDS[:3]:
    plot_event_gather(eid, component='Z', tmin=0, tmax=0.5)

## 8. Plot pick overlays on a selected gather

This uses the catalog pick table to overlay first-break picks on a simple fallback wiggle plot. This helps verify whether automated picks are sensible.

In [ ]:
def station_to_x_from_code(station):
    try:
        return float(str(station)) / 100.0
    except Exception:
        return np.nan

def plot_mseed_with_pick_overlays(event_id, component='Z', tmin=0.0, tmax=0.5, picker=None, phase=None):
    # Prefer 3C MSEED because picks can be overlaid by station code easily.
    sub_files = shot_gather_files[shot_gather_files['event_id'].eq(event_id)].copy()
    sub_mseed = sub_files[sub_files.get('file_type', '').eq('mseed')] if 'file_type' in sub_files.columns else pd.DataFrame()
    if sub_mseed.empty:
        print('No mseed gather for event:', event_id)
        return
    path = Path(sub_mseed.iloc[0]['file_path'])
    st = read(str(path)).select(channel=f'*{component}')
    if len(st) == 0:
        print('No component traces:', component)
        return
    st.sort(keys=['station'])
    npts = min(tr.stats.npts for tr in st)
    dt = st[0].stats.delta
    t = np.arange(npts) * dt
    mask = (t >= tmin) & (t <= tmax)

    fig, ax = plt.subplots(figsize=(13, 8))
    y_positions = {}
    for i, tr in enumerate(st):
        x = tr.data[:npts].astype(float)
        x = x - np.nanmedian(x)
        s = np.nanpercentile(np.abs(x), 99)
        if s > 0:
            x = x / s
        ax.plot(x[mask] * 0.35 + i, t[mask], linewidth=0.7)
        y_positions[tr.stats.station] = i

    p = picks[picks['event_id'].eq(event_id)].copy() if not picks.empty else pd.DataFrame()
    if not p.empty and 'component' in p.columns:
        p = p[p['component'].eq(component)]
    if picker is not None and 'picker' in p.columns:
        p = p[p['picker'].eq(picker)]
    if phase is not None and 'phase' in p.columns:
        p = p[p['phase'].eq(phase)]

    # Expected pick_time_relative_s column is preferred; otherwise infer from UTC.
    if not p.empty:
        if 'pick_time_relative_s' in p.columns:
            p['pick_rel_s'] = pd.to_numeric(p['pick_time_relative_s'], errors='coerce')
        elif 'pick_time_utc_dt' in p.columns:
            t0 = min(tr.stats.starttime for tr in st)
            p['pick_rel_s'] = p['pick_time_utc_dt'].apply(lambda x: UTCDateTime(x.to_pydatetime()) - t0 if pd.notna(x) else np.nan)
        else:
            p['pick_rel_s'] = np.nan

        for _, row in p.dropna(subset=['pick_rel_s']).iterrows():
            sta = str(row.get('station', ''))
            if sta in y_positions:
                ax.plot(y_positions[sta], row['pick_rel_s'], 'ro', markersize=3)

    ax.invert_yaxis()
    ax.set_xlabel('Trace index + normalized amplitude')
    ax.set_ylabel('Time since gather start (s)')
    ax.set_title(f'{event_id} {component} with pick overlays')
    ax.grid(True, alpha=0.2)
    plt.show()

if QC_EVENT_IDS:
    plot_mseed_with_pick_overlays(QC_EVENT_IDS[0], component='Z', tmin=0, tmax=0.5)

## 9. Common-shot metadata preview

This previews shot positions that appear in both nodal events and imported Geode/streamer metadata. It is a first step toward later cross-system comparison notebooks.

In [ ]:
if not shot_events.empty and {'instrument_system', 'source_x_m'}.issubset(shot_events.columns):
    se = shot_events.copy()
    se['source_x_round_m'] = pd.to_numeric(se['source_x_m'], errors='coerce').round(2)
    common = (
        se.dropna(subset=['source_x_round_m'])
        .groupby(['line', 'source_x_round_m', 'instrument_system'], dropna=False)
        .size()
        .reset_index(name='n')
    )
    pivot = common.pivot_table(
        index=['line', 'source_x_round_m'],
        columns='instrument_system',
        values='n',
        fill_value=0,
        aggfunc='sum',
    ).reset_index()
    display(pivot.sort_values(['line', 'source_x_round_m']).head(100))
    pivot.to_csv(QC_OUTPUT / 'common_source_position_preview.csv', index=False)
else:
    print('Need instrument_system and source_x_m in shot_events for common-shot preview.')

## 10. Export compact QC reports

Writes small CSV snapshots to `qc_outputs/` for review.

In [ ]:
# Compact exports
if not shot_events.empty:
    export_cols = [c for c in [
        'event_id', 'instrument_system', 'line', 'timewindow_label', 'geometry_id',
        'detection_time_utc', 'shot_time_utc', 'source_x_m', 'source_type', 'operator',
        'plate_type', 'n_blows', 'metadata_match_status', 'matched_metadata_event_id',
        'metadata_time_residual_s'
    ] if c in shot_events.columns]
    shot_events[export_cols].to_csv(QC_OUTPUT / 'shot_events_qc_snapshot.csv', index=False)

if not receiver_geometry.empty:
    receiver_geometry.to_csv(QC_OUTPUT / 'receiver_geometry_qc_snapshot.csv', index=False)

if not shot_gather_files.empty:
    shot_gather_files.to_csv(QC_OUTPUT / 'shot_gather_files_qc_snapshot.csv', index=False)

print('Wrote QC outputs to:', QC_OUTPUT)